In [ ]:
import os
import csv
import subprocess
import datetime

In [ ]:
# ---------- Configuration ----------
SOURCE_DIR = r"G:\thecall\done"
DEST_DIR   = r"e:\c_out"
SEVEN_ZIP_PATH = r"C:\Program Files\7-Zip\7z.exe"
CSV_PATH   = os.path.join(DEST_DIR, "inventory.csv")
ERROR_LOG  = os.path.join(DEST_DIR, "errors.log")

In [ ]:
def log_event(status, iso_path, message=""):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(ERROR_LOG, "a", encoding="utf-8") as f:
        f.write(f"{timestamp} | {status} | {iso_path} | {message}\n")

def clean_name(name):
    if not name: return "unnamed"
    invalid = '<>:"/\\|?*'
    for char in invalid:
        name = name.replace(char, "_")
    return name.strip().strip('.')

def extract_with_7zip(iso_path, out_folder):
    """
    Calls 7-Zip to extract the ISO. Returns True if successful.
    """
    try:
        # 7-Zip handles long paths well, so we don't necessarily need \\?\ here
        # but we use it for the output directory to be safe.
        cmd = [
            SEVEN_ZIP_PATH,
            "x",                 # Extract with paths
            iso_path,            # Source ISO
            f"-o{out_folder}",   # Output destination
            "-aos",              # Skip if file exists (prevents overwrite errors)
            "-y"                 # Assume 'Yes' to all queries
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            return True
        else:
            log_event("7Z_FAIL", iso_path, result.stderr)
            return False
    except Exception as e:
        log_event("7Z_EXCEPTION", iso_path, str(e))
        return False

def main():
    os.makedirs(DEST_DIR, exist_ok=True)
    
    # Fieldnames for inventory
    fieldnames = ["iso_file", "parent_folder", "status", "out_path"]

    with open(CSV_PATH, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for root, _, files in os.walk(SOURCE_DIR):
            parent_folder_name = os.path.basename(root)
            
            for file in files:
                if file.lower().endswith(".iso"):
                    iso_path = os.path.join(root, file)
                    iso_stem = os.path.splitext(file)[0]
                    
                    # Naming convention: {Number}_{ISO_Name}
                    folder_name = f"{parent_folder_name}_{iso_stem}"
                    out_folder = os.path.join(DEST_DIR, clean_name(folder_name))
                    
                    print(f"Extracting with 7-Zip: {parent_folder_name} -> {file}")
                    
                    success = extract_with_7zip(iso_path, out_folder)
                    
                    status = "SUCCESS" if success else "FAILED"
                    writer.writerow({
                        "iso_file": file,
                        "parent_folder": parent_folder_name,
                        "status": status,
                        "out_path": out_folder
                    })

    print(f"Batch process complete. Results logged to {CSV_PATH}")

if __name__ == "__main__":
    main()